# Ejercicio 4: Modelo Probabilístico

## Objetivo de la práctica
- Comprender los componentes del modelo vectorial mediante cálculos manuales y observación directa.
- Aplicar el modelo de espacio vectorial con TF-IDF para recuperar documentos relevantes.
- Comparar la recuperación con BM25 frente a TF-IDF.
- Analizar visualmente las diferencias entre los modelos.
- Evaluar si los rankings generados son consistentes con lo que considerarías documentos relevantes.

## Parte 0: Carga del Corpus

In [ ]:
from sklearn.datasets import fetch_20newsgroups

newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
newsgroupsdocs = newsgroups.data

## Parte 1: Cálculo de TF, DF, IDF y TF-IDF

### Actividad
1. Utiliza el corpus cargado.
2. Construye la matriz de términos (TF), y calcula la frecuencia de documentos (DF)
3. Calcula TF-IDF utilizando sklearn.
4. Visualiza los valores en un DataFrame para analizar las diferencias entre los términos.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


print("Número de documentos:", len(newsgroupsdocs))
#Verificar antes y despues de limpieza de texto
vectorizer_raw = TfidfVectorizer()
terms_raw = vectorizer_raw.fit(newsgroupsdocs).get_feature_names_out()

#Vectorizerize the documents
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    token_pattern=r'(?u)\b[a-záéíóúñü]{2,}\b'  # solo letras, mínimo 2 caracteres
)
corpusvect = vectorizer.fit_transform(newsgroupsdocs)
terms = vectorizer.get_feature_names_out()
print("Número de términos después de limpieza:", len(terms))
print("Términos sin limpieza:", len(terms_raw))


# Cada columna representa un término.
# corpusvect > 0 da True (1) donde el término aparece.
df_counts = (corpusvect > 0).sum(axis=0)  # matriz 1xN
df_array = np.asarray(df_counts).flatten()  # convertir a array 1D

# Asignar a variable df (como Serie de Pandas)
df = pd.Series(df_array, index=terms, name="Document Frequency")

print(df.sort_values(ascending=False).head(10))

#Pasar a dataframe
df_tfidf = pd.DataFrame.sparse.from_spmatrix(corpusvect, columns=terms)

print(df_tfidf.head())  # primeras filas

# Similaridad entre todos los documentos del corpus
similarity_matrix = cosine_similarity(corpusvect)



# Supongamos que quieres ver los 5 documentos más similares al documento 0
doc_id = 5
similarities = similarity_matrix[doc_id]

# Ordenar de mayor a menor similitud, ignorando el propio doc (índice 0)
similar_indices = similarities.argsort()[::-1][1:6]
print(f"Documentos más similares al documento {doc_id}: {similar_indices}")
print(f"Similitudes: {similarities[similar_indices]}")


Número de documentos: 18846
Número de términos después de limpieza: 86568
Términos sin limpieza: 134410
like      4212
just      4081
don       3894
know      3799
think     3144
does      2988
people    2988
time      2949
use       2705
good      2558
Name: Document Frequency, dtype: int64
   aa  aaa  aaaaa  aaaaaaaaaaaa  \
0   0    0      0             0   
1   0    0      0             0   
2   0    0      0             0   
3   0    0      0             0   
4   0    0      0             0   

   aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaauuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuugggggggggggggggg  \
0                                                  0                                 
1                                                  0                                 
2                                                  0                                 
3                                                  0                                 
4                                                  0              

In [ ]:
#Propiedades de la matriz de similitud de coseno
print("Tipo:", type(similarity_matrix))
print("Shape:", similarity_matrix.shape)
print("Ejemplo (fila 0):", similarity_matrix[0][:5])
print("Memoria usada (bytes):", similarity_matrix.nbytes)
print("Memoria usada (MB):", similarity_matrix.nbytes / (1024**2))
# Tipo de dato en cada celda
print("dtype:", similarity_matrix.dtype)

# Acceso rápido a máximos/mínimos
print("Similitud mínima:", similarity_matrix.min())
print("Similitud máxima:", similarity_matrix.max())

# ¿Diagonal 1s? (auto-similitud)
print("¿Diagonal toda 1?:", np.allclose(np.diag(similarity_matrix), 1.0))


Tipo: <class 'numpy.ndarray'>
Shape: (18846, 18846)
Ejemplo (fila 0): [1.         0.         0.02542152 0.         0.        ]
Memoria usada (bytes): 2841373728
Memoria usada (MB): 2709.745147705078
dtype: float64
Similitud mínima: 0.0
Similitud máxima: 1.0000000000000366
¿Diagonal toda 1?: False


## Parte 2: Ranking de documentos usando TF-IDF

### Actividad

1. Dada una consulta, construye el vector de consulta
2. Calcula la similitud coseno entre la consulta y cada documento usando los vectores TF-IDF
3. Genera un ranking de los documentos ordenados por relevancia.
4. Muestra los resultados en una tabla.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity



#Vectorizar la consulta
consulta_vec = vectorizer.transform([consulta])

# Calcular similitud del coseno entre la consulta y todos los documentos
similitudes = cosine_similarity(consulta_vec, corpusvect)  # Resultado: matriz 1 x n_docs

# Convertir a array plano y mostrar top resultados
similitudes = similitudes.flatten()

# Mostrar los documentos más similares
top_n = 5  # Cambia este valor si quieres más resultados
top_docs_idx = similitudes.argsort()[::-1][:top_n]

#filtrar a mostrar solo documentos done aparece la palabra
palabra_clave = input("Ingresa una palabra clave para filtrar: ")
print("\nTop documentos similares con la palabra clave:")
for i, idx in enumerate(top_docs_idx):
    if palabra_clave.lower() in newsgroupsdocs[idx].lower():
        print(f"\nDocumento #{idx} - Similitud: {similitudes[idx]:.4f}")
        print(newsgroupsdocs[idx][:500], "...")  # Muestra solo primeros 500 caracteres
#total de documentos donde aparece la palabra clave
total_documentos_con_palabra = sum(1 for idx in top_docs_idx if palabra_clave.lower() in newsgroupsdocs[idx].lower())
print(f"\nTotal de documentos con la palabra clave '{palabra_clave}': {total_documentos_con_palabra}")

Ingresa una palabra clave para filtrar: global

Top documentos similares con la palabra clave:

Total de documentos con la palabra clave 'global': 0


In [ ]:
#Propiedades del array despues de .flatten()
print("Tipo:", type(similitudes))
print("Shape:", similitudes.shape)
print("Ejemplo (elemento 0):", similitudes[0])
print("Memoria usada (bytes):", similitudes.nbytes)
print("Memoria usada (MB):", similitudes.nbytes / (1024**2))
# Tipo de dato en cada celda
print("dtype:", similitudes.dtype)

# Acceso rápido a máximos/mínimos
print("Similitud mínima:", similitudes.min())
print("Similitud máxima:", similitudes.max())




Tipo: <class 'numpy.ndarray'>
Shape: (18846,)
Ejemplo (elemento 0): 0.0
Memoria usada (bytes): 150768
Memoria usada (MB): 0.1437835693359375
dtype: float64
Similitud mínima: 0.0
Similitud máxima: 0.29967824246338237


## Parte 3: Ranking con BM25

### Actividad

1. Implementa un sistema de recuperación usando el modelo BM25.
2. Usa la misma consulta del ejercicio anterior.
3. Calcula el score BM25 para cada documento y genera un ranking.
4. Compara manualmente con el ranking de TF-IDF.

In [ ]:
!pip install rank_bm25
from rank_bm25 import BM25Okapi

In [ ]:
#implementacion de un sistema de recuperacion usando el modelo bm25

tokenized_corpus = [vectorizer.build_tokenizer()(doc) for doc in newsgroupsdocs]

# Initialize BM25 with the tokenized corpus
bm25_doc = BM25Okapi(tokenized_corpus)

# Tokenize the query as well before passing it to get_scores
consulta_tokenized = vectorizer.build_tokenizer()(consulta)
score=bm25_doc.get_scores(consulta_tokenized)
pd.DataFrame({'Document Index': range(len(newsgroupsdocs)), 'bm25score': score})
df.sort_values(ascending=False).head(10)

,Document Frequency
like,4212
just,4081
don,3894
know,3799
think,3144
does,2988
people,2988
time,2949
use,2705
good,2558


## Parte 4: Comparación visual entre TF-IDF y BM25

### Actividad

1. Utiliza un gráfico de barras para visualizar los scores obtenidos por cada documento según TF-IDF y BM25.
2. Compara los rankings visualmente.
3. Identifica: ¿Qué documentos obtienen scores más altos en un modelo que en otro?
4. Sugiere: ¿A qué se podría deber esta diferencia?

## Parte 5: Evaluación con consulta relevante

### Actividad

1. Elige una consulta y define qué documentos del corpus deberían considerarse relevantes.
2. Evalúa Precision@3 o MAP para los rankings generados con TF-IDF y BM25.
3. Responde: ¿Cuál modelo da mejores resultados respecto a tu criterio de relevancia?